# 🏥 Taranga AI: Technical Report — Data Synthesis

**Module:** `01_synthesizer`  
**Objective:** Create a statistically sound, comorbidity-aware dataset for training the Taranga primary screening model.

---

## 1. Methodology: The Synthetic Data Pipeline

In the context of learning-difficulty (LD) screening, high-quality labeled datasets are difficult to acquire due to clinical privacy (FERPA/HIPAA) and the high cost of expert assessment. Taranga addresses this by programmatically synthesizing a student population based on established epidemiological data.

### 1.1 Prevalence Parameters
We model five distinct learning difficulty domains with the following population prevalence rates:

- **Dyslexia:** 18%  
- **Dyscalculia:** 12%  
- **Dysgraphia:** 10%  
- **NVLD:** 8%  
- **APD:** 10%  

### 1.2 Comorbidity Logic (Overlaps)
A key clinical reality is that LDs rarely occur in isolation. Our generator implements dependency logic based on common educational patterns:
- **Dyslexia ↔ Dysgraphia:** 45% of dyslexic students also display dysgraphia indicators.
- **Dyslexia ↔ APD:** 25% overlap in phonological and auditory processing.
- **Dyscalculia ↔ NVLD:** 30% overlap in spatial and logic processing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set Premium Styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'

np.random.seed(42)

def generate_synthetic_data(num_samples=3000):
    PREV = {"dyslexia": 0.18, "dyscalculia": 0.12, "dysgraphia": 0.10, "nvld": 0.08, "apd": 0.10}
    data = []

    for _ in range(num_samples):
        # Binary labels
        is_dyslexia    = np.random.rand() < PREV["dyslexia"]
        is_dyscalculia = np.random.rand() < PREV["dyscalculia"]
        is_dysgraphia  = np.random.rand() < PREV["dysgraphia"]
        is_nvld        = np.random.rand() < PREV["nvld"]
        is_apd         = np.random.rand() < PREV["apd"]

        # Comorbidity logic
        if is_dyslexia and np.random.rand() < 0.45: is_dysgraphia = True
        if is_dyslexia and np.random.rand() < 0.25: is_apd = True
        if is_dyscalculia and np.random.rand() < 0.30: is_nvld = True

        def score(has_ld, low=(1, 4), high=(3, 6)): 
            return int(np.random.randint(*high)) if has_ld else int(np.random.randint(*low))

        row = {
            "q1_reading_pace": score(is_dyslexia), "q2_spelling_errors": score(is_dyslexia),
            "q3_letter_reversal": score(is_dyslexia), "q4_phonological": score(is_dyslexia),
            "q5_math_operations": score(is_dyscalculia), "q6_number_memory": score(is_dyscalculia),
            "q7_time_concept": score(is_dyscalculia), "q8_counting_patterns": score(is_dyscalculia),
            "q9_writing_quality": score(is_dysgraphia), "q10_pencil_grip": score(is_dysgraphia),
            "q11_word_spacing": score(is_dysgraphia), "q12_copying_accuracy": score(is_dysgraphia),
            "q13_social_cues": score(is_nvld), "q14_spatial_reasoning": score(is_nvld),
            "q15_routine_flexibility": score(is_nvld), "q16_nonverbal_comprehension": score(is_nvld),
            "q17_background_noise": score(is_apd), "q18_verbal_instructions": score(is_apd),
            "q19_sound_discrimination": score(is_apd), "q20_listening_retention": score(is_apd),
            "dyslexia": int(is_dyslexia), "dyscalculia": int(is_dyscalculia), 
            "dysgraphia": int(is_dysgraphia), "nvld": int(is_nvld), "apd": int(is_apd)
        }
        data.append(row)
    return pd.DataFrame(data)

df = generate_synthetic_data(3000)
print(f"Dataset successfully generated: {df.shape[0]} samples.")

## 2. Statistical Reports & Visualizations

### 2.1 Prevalence Breakdown
The donut chart below shows the distribution of identified LD indicators in the synthetic population.

In [ ]:
label_cols = ["dyslexia", "dyscalculia", "dysgraphia", "nvld", "apd"]
counts = df[label_cols].sum()

plt.figure(figsize=(10, 6))
plt.pie(counts, labels=[c.upper() for c in label_cols], autopct='%1.1f%%', 
        startangle=140, colors=sns.color_palette("viridis", 5), pctdistance=0.85, 
        explode=[0.05]*5)

# Add center circle for donut effect
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title("Synthetic Dataset: LD Prevalence Breakdown", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.2 Feature Score Distributions
We analysis the distribution of scores (1–5) across all observations. In a balanced dataset, we expect a multi-modal distribution where higher scores correspond to positive LD flags.

In [ ]:
feat_cols = [f for f in df.columns if f.startswith('q')]
sample_feats = feat_cols[:4] # Display first 4 (Dyslexia features)

plt.figure(figsize=(16, 4))
for i, feat in enumerate(sample_feats):
    plt.subplot(1, 4, i+1)
    sns.histplot(df[feat], bins=5, kde=True, color='#6366F1')
    plt.title(f"{feat.replace('_', ' ').title()}", fontsize=10)
    plt.xticks([1, 2, 3, 4, 5])

plt.suptitle("Observation Score Density (Sample: Dyslexia Indicators)", fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### 2.3 Feature Correlation Heatmap
A strong diagnostic model relies on features within a domain being highly correlated with each other while remaining distinct from others. The heatmap shows our successful grouping of 4-questions per LD.

In [ ]:
plt.figure(figsize=(14, 10))
corr = df[feat_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="mako", annot=False, linewidths=.5)
plt.title("Inter-Question Correlation Matrix (Identifying Clustering)", fontsize=15, fontweight='bold')
plt.show()

### 2.4 Comorbidity (Overlap) Report
Below is the statistical report on students with multi-difficulty profiles.

In [ ]:
overlaps = df[label_cols].sum(axis=1)
overlap_counts = overlaps.value_counts().sort_index()

report = pd.DataFrame({
    "LD Count": [f"{i} Conditions" for i in overlap_counts.index],
    "Student Count": overlap_counts.values,
    "Percentage": [f"{v/3000:.1%}" for v in overlap_counts.values]
})
print("COMORBIDITY ANALYSIS REPORT")
print("=" * 30)
print(report.to_string(index=False))